In [1]:
!pip install scikit-learn imbalanced-learn xgboost joblib pandas numpy -q
print('✓ Library siap')

✓ Library siap


In [2]:
# mount google drive & setup path

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Path ke folder project di Google Drive ──
PROJECT_DIR  = '/content/drive/MyDrive/aquaculture_ml'
DATASET_DIR  = os.path.join(PROJECT_DIR, 'datasets')
MODEL_DIR    = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
SRC_DIR      = os.path.join(PROJECT_DIR, 'src')

for d in [DATASET_DIR, MODEL_DIR, OUTPUT_DIR, SRC_DIR]:
    os.makedirs(d, exist_ok=True)

# Tambahkan src/ ke Python path supaya bisa import modul
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'✓ Google Drive mounted')
print(f'✓ Project dir : {PROJECT_DIR}')
print(f'✓ Src dir added to path')

Mounted at /content/drive
✓ Google Drive mounted
✓ Project dir : /content/drive/MyDrive/aquaculture_ml
✓ Src dir added to path


In [3]:
# upload dataset

from google.colab import files
import shutil

print('Upload realfishdataset.csv:')
uploaded = files.upload()
for fname in uploaded:
    dest = os.path.join(DATASET_DIR, fname)
    shutil.move(fname, dest)
    print(f'  ✓ {fname} → {dest}')

print('\nUpload pondsdata.csv:')
uploaded2 = files.upload()
for fname in uploaded2:
    dest = os.path.join(DATASET_DIR, fname)
    shutil.move(fname, dest)
    print(f'  ✓ {fname} → {dest}')

Upload realfishdataset.csv:


Saving realfishdataset.csv to realfishdataset.csv
  ✓ realfishdataset.csv → /content/drive/MyDrive/aquaculture_ml/datasets/realfishdataset.csv

Upload pondsdata.csv:


Saving Aquaponds Dataset.csv to Aquaponds Dataset.csv
  ✓ Aquaponds Dataset.csv → /content/drive/MyDrive/aquaculture_ml/datasets/Aquaponds Dataset.csv


In [4]:
# TRAINING - MODEL
# Jalankan train_model.py dari src/
# Model akan disimpan ke models/ di Google Drive

from train_model import main as train_main
train_main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  AQUACULTURE ML — TRAINING PIPELINE

[1/6] Loading dataset dari: /content/drive/MyDrive/aquaculture_ml/datasets/realfishdataset.csv
      ✓ Dataset loaded: 40280 baris, 4 kolom
      ✓ Distribusi kelas:
fish
tilapia      8830
rui          6336
pangas       5314
silverCup    3906
katla        3786
sing         3776
shrimp       3204
karpio       2112
prawn        1348
koi           964
magur         704

[2/6] Preprocessing...
      ✓ Kelas yang ditemukan: ['karpio', 'katla', 'koi', 'magur', 'pangas', 'prawn', 'rui', 'shrimp', 'silverCup', 'sing', 'tilapia']
      ✓ Jumlah kelas: 11

[3/6] Menerapkan SMOTE untuk menangani class imbalance...
      ✓ Sebelum SMOTE: 32224 sampel
      ✓ Sesudah SMOTE : 77704 sampel

[4/6] Training Random Forest dengan GridSearchCV...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

      ✓ Best params : {'max_dept

In [5]:
# Inisialisasi IoT Simulator

import json
from simulate_iot import init_iot_simulator, get_all_stations, get_station_by_id, get_station_statistics

# Load Pondsdata sekali di sini
pond_df = init_iot_simulator(
    path=os.path.join(DATASET_DIR, 'pondsdata.csv')
)

# Lihat semua stasiun yang tersedia
all_stations = get_all_stations(pond_df)
print('=== Daftar Stasiun IoT ===')
print(json.dumps(all_stations, indent=2))

✓ IoT Simulator ready — 49998 records dari 3 stasiun
  Distribusi per stasiun:
station_id
pond_1    16666
pond_2    16666
pond_3    16666

=== Daftar Stasiun IoT ===
[
  {
    "station_id": "pond_1",
    "name": "Pond Station 1",
    "description": "Kolam ikan murrel \u2014 Guntur, Andhra Pradesh",
    "fish_stock": "Murrel",
    "water_condition": {
      "ph": 5.94,
      "temperature": 26.04,
      "turbidity": 34.64
    }
  },
  {
    "station_id": "pond_2",
    "name": "Pond Station 2",
    "description": "Kolam ikan catla \u2014 Guntur, Andhra Pradesh",
    "fish_stock": "Catla",
    "water_condition": {
      "ph": 6.0,
      "temperature": 28.86,
      "turbidity": 23.8
    }
  },
  {
    "station_id": "pond_3",
    "name": "Pond Station 3",
    "description": "Kolam multispecies \u2014 Guntur, Andhra Pradesh",
    "fish_stock": "Multispecies",
    "water_condition": {
      "ph": 5.32,
      "temperature": 27.43,
      "turbidity": 28.67
    }
  }
]


In [6]:
# Load Model & evalusi ikan

from evaluate_fish import load_model, evaluate_fish, save_result

# Load model
model, le = load_model()

✓ Model loaded — 11 kelas ikan: ['karpio', 'katla', 'koi', 'magur', 'pangas', 'prawn', 'rui', 'shrimp', 'silverCup', 'sing', 'tilapia']



In [7]:
# ── Simulasi input dari FE ──────────────────────────────────────
# Di sistem nyata, nilai ini dikirim dari FE via API request ke BE

SELECTED_STATION = 'pond_1'     # user klik titik kolam di peta
SELECTED_FISH    = 'Tilapia'    # user drag & drop ikan

# Ambil kondisi air di titik yang dipilih
station_info = get_station_by_id(pond_df, SELECTED_STATION)
print(f'Kondisi air {SELECTED_STATION}:')
print(json.dumps(station_info['water_condition'], indent=2))

# Evaluasi ikan
result = evaluate_fish(
    model            = model,
    le               = le,
    fish_name        = SELECTED_FISH,
    water_condition  = station_info['water_condition'],
    station_info     = station_info
)

# Tampilkan hasil (ini yang dikirim ke FE/BE)
print('\n=== RESPONSE KE FE/BE ===')
print(json.dumps(result, indent=2))

# Simpan ke outputs/
save_result(result, 'evaluation_result.json')

Kondisi air pond_1:
{
  "ph": 5.94,
  "temperature": 26.04,
  "turbidity": 34.64
}

=== RESPONSE KE FE/BE ===
{
  "status": "error",
  "timestamp": "2026-06-17T05:52:32.790046+00:00",
  "error_code": "FISH_NOT_FOUND",
  "message": "Ikan 'Tilapia' tidak ada dalam database model. Ikan yang tersedia: ['karpio', 'katla', 'koi', 'magur', 'pangas', 'prawn', 'rui', 'shrimp', 'silverCup', 'sing', 'tilapia']",
  "fish_queried": "Tilapia",
  "station_id": "pond_1"
}
✓ Hasil disimpan: /content/drive/MyDrive/aquaculture_ml/outputs/evaluation_result.json


'/content/drive/MyDrive/aquaculture_ml/outputs/evaluation_result.json'

In [8]:
# uji semua kombinasi titik x ikan

import pandas as pd

# Test semua kombinasi titik × ikan → lihat hasilnya dalam tabel
stations_to_test = ['pond_1', 'pond_2', 'pond_3']
fish_list        = list(le.classes_)

rows = []
for sid in stations_to_test:
    s = get_station_by_id(pond_df, sid)
    wc = s['water_condition']
    for fish in fish_list:
        r = evaluate_fish(model, le, fish, wc, s)
        if r['status'] == 'success':
            ev = r['evaluation']
            rows.append({
                'station'       : sid,
                'fish'          : fish,
                'recommendation': ev['recommendation'],
                'confidence'    : ev['confidence_score'],
                'label'         : ev['confidence_label'],
            })

df_results = pd.DataFrame(rows).sort_values(['station', 'confidence'], ascending=[True, False])
print(df_results.to_string(index=False))

station      fish  recommendation  confidence    label
 pond_1    karpio        possible        0.36      Low
 pond_1   tilapia not_recommended        0.26      Low
 pond_1    shrimp not_recommended        0.17 Very Low
 pond_1      sing not_recommended        0.15 Very Low
 pond_1       koi not_recommended        0.06 Very Low
 pond_1     katla not_recommended        0.00 Very Low
 pond_1     magur not_recommended        0.00 Very Low
 pond_1    pangas not_recommended        0.00 Very Low
 pond_1     prawn not_recommended        0.00 Very Low
 pond_1       rui not_recommended        0.00 Very Low
 pond_1 silverCup not_recommended        0.00 Very Low
 pond_2    karpio not_recommended        0.31      Low
 pond_2      sing not_recommended        0.28      Low
 pond_2    shrimp not_recommended        0.19 Very Low
 pond_2   tilapia not_recommended        0.08 Very Low
 pond_2       rui not_recommended        0.06 Very Low
 pond_2       koi not_recommended        0.05 Very Low
 pond_2   

In [9]:
# Lihat statistik historis tiap titik
for sid in ['pond_1', 'pond_2', 'pond_3']:
    stats = get_station_statistics(pond_df, sid)
    print(f'\n=== Statistik {sid} ===')
    print(json.dumps(stats, indent=2))


=== Statistik pond_1 ===
{
  "ph": {
    "mean": 6.54,
    "min": 4.5,
    "max": 9.01,
    "std": 1.11
  },
  "temperature": {
    "mean": 27.25,
    "min": 0.0,
    "max": 45.48,
    "std": 5.96
  },
  "turbidity": {
    "mean": 31.63,
    "min": 10.2,
    "max": 81.0,
    "std": 11.66
  }
}

=== Statistik pond_2 ===
{
  "ph": {
    "mean": 6.55,
    "min": 4.5,
    "max": 9.01,
    "std": 1.11
  },
  "temperature": {
    "mean": 27.18,
    "min": 0.0,
    "max": 45.47,
    "std": 5.99
  },
  "turbidity": {
    "mean": 31.7,
    "min": 10.2,
    "max": 80.28,
    "std": 11.66
  }
}

=== Statistik pond_3 ===
{
  "ph": {
    "mean": 6.55,
    "min": 4.5,
    "max": 9.01,
    "std": 1.11
  },
  "temperature": {
    "mean": 27.19,
    "min": 0.0,
    "max": 45.46,
    "std": 5.95
  },
  "turbidity": {
    "mean": 31.5,
    "min": 10.2,
    "max": 81.55,
    "std": 11.57
  }
}
